In [ ]:
library(Seurat)
library(ggplot2)
library(org.Hs.eg.db)
library(GOSemSim)
library(clusterProfiler)

In [ ]:
data <- readRDS("/data/work/sc_reference/EXN_harmony_nasa_res0_5_annotated.rds")
data <- subset(data, subset = annotation != "Other")
table(data@meta.data$annotation)

# Gene symbols must be mapped to the human OrgDb before enrichment.
valid_symbols <- AnnotationDbi::keys(org.Hs.eg.db, keytype = "SYMBOL")
counts <- GetAssayData(data, assay = "RNA", slot = "counts")
expressed <- rownames(counts)[Matrix::rowSums(counts) > 0]
universe <- intersect(expressed, valid_symbols)
stopifnot(length(universe) > 0L)

In [ ]:
## 提取差异基因进行富集

In [ ]:
deg_results <- read.csv("/data/work/08_GO_analysis/deg_nasa_ex_annotation_ratio_filtered_0_5.csv")
ex_bdnf_genes <- deg_results[deg_results$cluster == "EX_BDNF", "gene"]
ex_bdnf_genes <- intersect(unique(ex_bdnf_genes), universe)

In [ ]:
erich.go.BP = enrichGO(gene =ex_bdnf_genes,
                       OrgDb = org.Hs.eg.db,
                       keyType = "SYMBOL",
                       ont = "BP",
                       universe = universe,
                       pAdjustMethod = "BH",
                       pvalueCutoff = 0.05,
                       qvalueCutoff = 0.05)

dotplot(erich.go.BP) + ggtitle("GO Enrichment for EX_BDNF DEGs")
write.csv(as.data.frame(erich.go.BP), "GO_enrich_BP_for_EX_BDNF.csv", row.names = FALSE)

In [ ]:
## 循环输出每个兴奋性

In [ ]:
clusters <- c("EX_ADRA1A_Vat1l", "EX_BDNF", "EX_C1QL3", "EX_env_TMEM144", "EX_ERG",
              "EX_FOXP2", "EX_HTR7_GNAL", "EX_IFITM3_CLDN5", "EX_ITGA4_TPBG", "EX_NTNG1",
              "EX_Ntng2", "EX_Nxph4_CPLX3", "EX_SLC1A3_FGFR3", "EX_TSHZ2_VWC2L")

# 遍历每个兴奋性亚群
for (cluster in clusters) {
  # 提取当前亚群的基因
  genes <- deg_results[deg_results$cluster == cluster, "gene"]
  genes <- intersect(unique(genes), universe)
  
  # 检查基因列表是否为空
  if (length(genes) == 0) {
    cat("No genes found for cluster:", cluster, "\n")
    next
  }
  
  # 进行GO分析
  go_result <- enrichGO(gene = genes,
                        OrgDb = org.Hs.eg.db,
                        keyType = "SYMBOL",
                        ont = "BP",
                        universe = universe,
                        pAdjustMethod = "BH",
                        pvalueCutoff = 0.05,
                        qvalueCutoff = 0.05)
  
  # 检查GO分析结果是否为空
  if (is.null(go_result@result) || nrow(go_result@result) == 0) {
    cat("No significant GO terms found for cluster:", cluster, "\n")
    next
  }
  
  # 绘制点图并去掉标题
  tryCatch({
    p <- dotplot(go_result)  # 不添加标题
    
    # 保存为PNG和PDF格式
    ggsave(filename = paste(cluster, "_GO_enrichment_dotplot.png"), plot = p, width = 8, height = 8)
    ggsave(filename = paste(cluster, "_GO_enrichment_dotplot.pdf"), plot = p, width = 8, height = 8)
    
    # 输出循环完成的提示
    cat("Processing for cluster", cluster, "completed.\n")
  }, error = function(e) {
    cat("Error generating dotplot for cluster", cluster, ":\n", e$message, "\n")
  })
  
  # 保存GO分析结果到CSV文件
  write.csv(as.data.frame(go_result), paste(cluster, "_GO_enrich_BP.csv"), row.names = FALSE)
}